# Sommelier — Workflow 2.0 (Step-by-Step Ephemeral Execution)

This notebook implements **Workflow 2.0**. It divides the pipeline into 3 distinct phases:
1. **Diarization**: Setup environment, run Diarization only, then cleanup HDD/RAM.
2. **Separation**: Setup environment, run Separation only, then cleanup HDD/RAM.
3. **ASR & Export**: Setup environment, run ASR to finish, then cleanup.

By omitting flags like `--ASRMoE`, `--tse`, `--panns` in the early steps, we prevent the python script from downloading those heavy models early, saving massive amounts of HDD space without changing the original python source code.

In [ ]:
import os
os.environ["MPLBACKEND"] = "Agg"
!nvidia-smi
!df -h /kaggle/working 2>/dev/null || df -h .

## 0. Common Setup (Audio & Repo)

In [ ]:
import os
import shutil
import sys

BASE_DIR = '/kaggle/working' if os.path.exists('/kaggle/working') else os.getcwd()
PROJECT_DIR = os.path.join(BASE_DIR, 'sommerlier')
AUDIO_DIR = os.path.join(BASE_DIR, 'vi_audio')
os.makedirs(AUDIO_DIR, exist_ok=True)

# Set HuggingFace Hub to /tmp to prevent /kaggle/working Disk Full
os.environ["HF_HOME"] = "/tmp/huggingface"
os.environ["TORCH_HOME"] = "/tmp/torch"

# Clone Repo if not exists
if not os.path.exists(PROJECT_DIR):
    os.chdir(BASE_DIR)
    !git clone -b solid-architecture https://github.com/foresst123/sommerlier.git

# Make synthetic audio if folder is empty
import glob
if not glob.glob(os.path.join(AUDIO_DIR, '*.mp3')) and not glob.glob(os.path.join(AUDIO_DIR, '*.wav')):
    print("Please upload audio to /kaggle/working/vi_audio/")

---
## PHASE 1: DIARIZATION ONLY
Installs the base + diarizen environments. Runs `main.py` with `--stop_after diarization` and **without** `--tse`, `--panns`, or `--ASRMoE` flags so it doesn't download those heavy models yet.

In [ ]:
%%bash
BASE_DIR="/kaggle/working"
ENV_DIR="$BASE_DIR/sommelier_env"
DIARIZEN_ENV="$BASE_DIR/diarizen_env"

# 1. Setup Sommelier Env (Base)
uv venv --allow-existing $ENV_DIR
uv pip install --python $ENV_DIR torch==2.7.1 torchaudio==2.7.1 torchvision==0.22.1 --extra-index-url https://download.pytorch.org/whl/cu126
uv pip install --python $ENV_DIR numpy==2.2.2 librosa==0.10.2.post1 soundfile pydub pandas PyYAML tqdm requests huggingface-hub pyannote.audio==4.0.7 speechbrain==1.0.2 transformers==4.53.0

# 2. Setup DiariZen Env
uv venv --allow-existing $DIARIZEN_ENV --python 3.12
uv pip install --python $DIARIZEN_ENV torch==2.5.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121
uv pip install --python $DIARIZEN_ENV git+https://github.com/BUTSpeechFIT/DiariZen.git@844f5555b0a98acd0931511fc641a8c5b8ba92c7
uv pip install --python $DIARIZEN_ENV "git+https://github.com/BUTSpeechFIT/DiariZen.git@844f5555b0a98acd0931511fc641a8c5b8ba92c7#subdirectory=pyannote-audio" speechbrain==1.0.2 toml==0.10.2 wrapt==2.3.0 pyparsing matplotlib==3.9.4

echo "✅ Phase 1 Environments Ready!"

In [ ]:
import os
import sys

os.chdir(os.path.join(PROJECT_DIR, 'podcast-pipeline'))
python_bin = os.path.join(BASE_DIR, 'sommelier_env', 'bin', 'python')

!for f in {AUDIO_DIR}/*; do \
    if [ -f "$$f" ]; then \
        filename=$$(basename -- "$$f"); \
        job_id="$${filename%.*}"; \
        export DIARIZEN_PYTHON="/kaggle/working/diarizen_env/bin/python"; \
        CUDA_VISIBLE_DEVICES=0,1 {python_bin} main.py \
        --audio "$$f" \
        --job_id "$$job_id" \
        --stop_after diarization \
        --lang vi \
        --seg_th 0.11 \
        --min_cluster_size 11 \
        --clust_th 0.5; \
    fi; \
done
print("✅ Phase 1 Diarization Completed and Cached!")

In [ ]:
# 🧹 CLEANUP PHASE 1 (Free HDD & RAM)
!rm -rf /kaggle/working/diarizen_env
!rm -rf /tmp/huggingface/hub/*
print("✅ Phase 1 Cleanup Completed! HDD Space Recovered.")

---
## PHASE 2: SEPARATION & MUSIC REMOVAL
Installs the heavy models for separation (Demucs, PANNS). Runs `main.py` with `--stop_after music_removal` and re-adds the `--tse` and `--panns` flags.

In [ ]:
# Separation models are already installed in sommelier_env
print('✅ Phase 2 Dependencies Ready!')


In [ ]:
os.chdir(os.path.join(PROJECT_DIR, 'podcast-pipeline'))

!for f in {AUDIO_DIR}/*; do \
    if [ -f "$$f" ]; then \
        filename=$$(basename -- "$$f"); \
        job_id="$${filename%.*}"; \
        CUDA_VISIBLE_DEVICES=0,1 {python_bin} main.py \
        --audio "$$f" \
        --job_id "$$job_id" \
        --stop_after music_removal \
        --dia3 \
        --lang vi \
        --tse \
        --panns \
        --seg_th 0.11 \
        --min_cluster_size 11 \
        --clust_th 0.5; \
    fi; \
done
print("✅ Phase 2 Separation Completed and Cached!")

In [ ]:
# 🧹 CLEANUP PHASE 2
!rm -rf /tmp/huggingface/hub/*
print("✅ Phase 2 Cleanup Completed! HDD Space Recovered.")

---
## PHASE 3: ASR & EXPORT
Installs the massive ASR MoE models (Whisper, Qwen3). Runs pipeline to completion without `--stop_after`.

In [ ]:
%%bash
BASE_DIR="/kaggle/working"
QWEN3_ENV="$BASE_DIR/qwen3_env"
# ASR models are already in base env. Just setup Qwen3 Env
uv venv --allow-existing $QWEN3_ENV --python 3.12
uv pip install --python $QWEN3_ENV torch==2.7.1 torchaudio==2.7.1 torchvision==0.22.1 --extra-index-url https://download.pytorch.org/whl/cu126
uv pip install --python $QWEN3_ENV "transformers>=5.13.0" "huggingface-hub>=1.5.0" accelerate soundfile librosa
echo "✅ Phase 3 Environments Ready!"


In [ ]:
os.chdir(os.path.join(PROJECT_DIR, 'podcast-pipeline'))

!for f in {AUDIO_DIR}/*; do \
    if [ -f "$$f" ]; then \
        filename=$$(basename -- "$$f"); \
        job_id="$${filename%.*}"; \
        export QWEN3_PYTHON="/kaggle/working/qwen3_env/bin/python"; \
        CUDA_VISIBLE_DEVICES=0,1 {python_bin} main.py \
        --audio "$$f" \
        --job_id "$$job_id" \
        --lang vi \
        --tse \
        --panns \
        --ASRMoE \
        --dia3 \
        --seg_th 0.11 \
        --min_cluster_size 11 \
        --clust_th 0.5; \
    fi; \
done
print("✅ Phase 3 ASR Completed! Check the _final folder for outputs.")

In [ ]:
# Export Results
!zip -r /kaggle/working/vi_audio_output.zip /kaggle/working/vi_audio/_final

# 🧹 FINAL CLEANUP
!rm -rf /kaggle/working/qwen3_env
!rm -rf /tmp/huggingface/hub/*
print("✅ All Done!")